In [18]:
import yfinance as yf
import pandas as pd
import pandas_ta as ta
import numpy as np

In [19]:
ticker = "QQQ"
print(f"Fetching Raw Data for {ticker}")

df = yf.download(ticker, period="1y", interval="1h", progress=False)

print(f"Initial shape: {df.shape}")
print("---------------------------------------")
print("First 3 rows of raw data:")
print(df.head(3))


Fetching Raw Data for QQQ
Initial shape: (1745, 5)
---------------------------------------
First 3 rows of raw data:
Price                           Close        High         Low        Open  \
Ticker                            QQQ         QQQ         QQQ         QQQ   
Datetime                                                                    
2025-05-09 13:30:00+00:00  488.000092  491.540009  487.510101  490.279999   
2025-05-09 14:30:00+00:00  487.989990  489.000000  486.209991  488.040009   
2025-05-09 15:30:00+00:00  487.189911  488.959991  486.630005  488.000000   

Price                       Volume  
Ticker                         QQQ  
Datetime                            
2025-05-09 13:30:00+00:00  6310013  
2025-05-09 14:30:00+00:00  5260414  
2025-05-09 15:30:00+00:00  3043960  


In [20]:
print("Data Quality Check")

print(df.isna().sum())

# Drop the MultiIndex columns yfinance creates
df.columns = df.columns.droplevel(1) if isinstance(df.columns, pd.MultiIndex) else df.columns
# make columns lowercase
df.columns = [c.lower() for c in df.columns]


Data Quality Check
Price   Ticker
Close   QQQ       0
High    QQQ       0
Low     QQQ       0
Open    QQQ       0
Volume  QQQ       0
dtype: int64


In [21]:
print("Experimenting with Features")

# 1. Log Returns
df['log_return'] = np.log(df['close'] / df['close'].shift(1))

# 2. RSI (Momentum - is the stock overbought or oversold?)
df.ta.rsi(length=14, append=True)

# 3. MACD (Trend direction)
df.ta.macd(fast=12, slow=26, signal=9, append=True)

print([col for col in df.columns if 'RSI' in col or 'MACD' in col])

Experimenting with Features
['RSI_14', 'MACD_12_26_9', 'MACDh_12_26_9', 'MACDs_12_26_9']


In [22]:
print("Creating the Target Label")
# predict the Volatility (Standard Deviation) of the next 6 hours
horizon = 6
df['target_volatility_6h'] = df['log_return'].shift(-horizon).rolling(window=horizon).std()

# Drop rows that don't have enough data to calculate these rolling metrics
clean_df = df.dropna().copy()

print(f"Usable rows after dropping NaNs: {len(clean_df)}")
print("Summary Statistics of our Target (Volatility):")
print(clean_df['target_volatility_6h'].describe())


Creating the Target Label
Usable rows after dropping NaNs: 1706
Summary Statistics of our Target (Volatility):
count    1706.000000
mean        0.003405
std         0.002179
min         0.000204
25%         0.001759
50%         0.002835
75%         0.004613
max         0.014519
Name: target_volatility_6h, dtype: float64


In [23]:
print("Feature Correlation to Target")
# Which features actually matter for predicting volatility?
# A correlation of 1.0 means perfect alignment. 0 means useless.

features_to_check = [
    'log_return', 
    'RSI_14', 
    'MACD_12_26_9', 
    'MACDh_12_26_9', # MACD Histogram
    'volume'
]

for feature in features_to_check:
    corr = clean_df[feature].corr(clean_df['target_volatility_6h'])
    print(f"Correlation between {feature} and Target: {corr:.4f}")

Feature Correlation to Target
Correlation between log_return and Target: -0.0984
Correlation between RSI_14 and Target: -0.4041
Correlation between MACD_12_26_9 and Target: -0.3685
Correlation between MACDh_12_26_9 and Target: -0.2210
Correlation between volume and Target: 0.0606


In [24]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import math

In [25]:
print("Training XGBoost Model")

X = clean_df[features_to_check]
y = clean_df['target_volatility_6h']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

print(f"Training on {len(X_train)} rows, Testing on {len(X_test)} rows...")

model = xgb.XGBRegressor(
    n_estimators=100, 
    learning_rate=0.1, 
    max_depth=5, 
    random_state=42
)
model.fit(X_train, y_train)

predictions = model.predict(X_test)
rmse = math.sqrt(mean_squared_error(y_test, predictions))

print(f"RMSE: {rmse:.5f}")

Training XGBoost Model
Training on 1364 rows, Testing on 342 rows...
RMSE: 0.00240
